In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
!pip install gymnasium
!pip install stable-baselines3

In [ ]:
import gymnasium as gym

from gymnasium import spaces

import numpy as np

import matplotlib.pyplot as plt

from stable_baselines3 import PPO

from stable_baselines3.common.torch_layers import BaseFeaturesExtractor

import torch
import torch.nn as nn

In [ ]:
class CustomCNN(BaseFeaturesExtractor):

    def __init__(self, observation_space, features_dim=256):

        super(CustomCNN, self).__init__(
            observation_space,
            features_dim
        )

        n_input_channels = observation_space.shape[0]

        self.cnn = nn.Sequential(

            nn.Conv2d(
                n_input_channels,
                32,
                kernel_size=3,
                stride=1,
                padding=1
            ),

            nn.ReLU(),

            nn.Conv2d(
                32,
                64,
                kernel_size=3,
                stride=1,
                padding=1
            ),

            nn.ReLU(),

            nn.Flatten()
        )

        with torch.no_grad():

            sample_input = torch.as_tensor(
                observation_space.sample()[None]
            ).float()

            n_flatten = self.cnn(sample_input).shape[1]

        self.linear = nn.Sequential(

            nn.Linear(n_flatten, features_dim),

            nn.ReLU()
        )

    def forward(self, observations):

        return self.linear(
            self.cnn(observations)
        )

In [ ]:
class MultiAgentSaudiEnv(gym.Env):

    def __init__(self, num_agents=3):

        super(MultiAgentSaudiEnv, self).__init__()

        self.num_agents = num_agents

        # =====================================
        # LOAD SAUDI TENSOR
        # =====================================

        tensor_path = (
            "/content/drive/MyDrive/"
            "PyroRL_Saudi_Project/datasets/"
            "saudi_eastern_province/grids/32x32/"
            "state_tensor.npy"
        )

        self.initial_state = np.load(tensor_path)
        self.state = self.initial_state.copy()

        # =====================================
        # OBSERVATION SPACE
        # =====================================
        # 7 env channels + num_agents channels
        self.num_env_channels = 7
        total_channels = self.num_env_channels + self.num_agents

        self.observation_space = spaces.Box(
            low=0,
            high=1,
            shape=(total_channels, 32, 32),
            dtype=np.float32
        )

        # =====================================
        # MULTI-AGENT ACTION SPACE
        # =====================================

        self.action_space = spaces.MultiDiscrete(
            [5] * self.num_agents
        )

        self.current_step = 0
        self.max_steps = 200
        self.previous_mean_fire = None

        self.reset_agent_positions()

    def reset_agent_positions(self):

        self.agent_positions = []

        for _ in range(self.num_agents):
            x = np.random.randint(0, 32)
            y = np.random.randint(0, 32)
            self.agent_positions.append([x, y])

    def _get_obs(self):
        obs = np.zeros((self.num_env_channels + self.num_agents, 32, 32), dtype=np.float32)
        # Fill env channels
        obs[:self.num_env_channels, :, :] = self.state[:self.num_env_channels, :, :]
        # Fill agent position channels
        for idx, (x, y) in enumerate(self.agent_positions):
            obs[self.num_env_channels + idx, x, y] = 1.0
        return obs

    def reset(self, seed=None, options=None):

        super().reset(seed=seed)

        self.state = self.initial_state.copy()
        self.current_step = 0

        self.reset_agent_positions()
        self.previous_mean_fire = np.mean(self.state[0])

        return self._get_obs(), {}


    def step(self, actions):

        fire_layer = self.state[0]
        fuel_layer = self.state[1]
        wind_x = self.state[2]
        wind_y = self.state[3]
        terrain_layer = self.state[4]

        # =====================================
        # AGENT MOVEMENT
        # =====================================

        for idx, action in enumerate(actions):

            x, y = self.agent_positions[idx]

            if action == 0:
                x -= 1
            elif action == 1:
                x += 1
            elif action == 2:
                y -= 1
            elif action == 3:
                y += 1

            x = np.clip(x, 0, 31)
            y = np.clip(y, 0, 31)

            self.agent_positions[idx] = [x, y]

        # =====================================
        # SPATIAL FIRE SPREAD
        # =====================================

        new_fire_layer = fire_layer.copy()

        for i in range(1, 31):
            for j in range(1, 31):
                current_fire = fire_layer[i, j]

                if current_fire < 0.05:
                    continue

                neighbors = [
                    (i-1, j),
                    (i+1, j),
                    (i, j-1),
                    (i, j+1)
                ]

                for ni, nj in neighbors:
                    fuel = fuel_layer[ni, nj]

                    wind_bonus = abs(wind_x[ni, nj]) + abs(wind_y[ni, nj])
                    terrain_factor = 1 + terrain_layer[ni, nj]

                    spread_amount = (
                        0.02
                        * current_fire
                        * fuel
                        * (1 + wind_bonus)
                        * terrain_factor
                    )

                    new_fire_layer[ni, nj] += spread_amount

        fire_layer = np.clip(new_fire_layer, 0, 1)

        # =====================================
        # MULTI-AGENT SUPPRESSION (3x3 RADIUS)
        # =====================================

        suppression_power = 0.05

        for x, y in self.agent_positions:
            for dx in [-1, 0, 1]:
                for dy in [-1, 0, 1]:
                    nx, ny = x + dx, y + dy
                    if 0 <= nx < 32 and 0 <= ny < 32:
                        fire_layer[nx, ny] = max(
                            0.0,
                            fire_layer[nx, ny] - suppression_power
                        )

        # =====================================
        # FUEL CONSUMPTION
        # =====================================

        fuel_layer = np.clip(
            fuel_layer - 0.01 * fire_layer,
            0,
            1
        )

        self.state[1] = fuel_layer

        # =====================================
        # REWARD
        # =====================================

        current_mean_fire = np.mean(fire_layer)
        reward = -current_mean_fire

        # Delta reward: Positive reward if mean fire decreases
        fire_delta = self.previous_mean_fire - current_mean_fire
        if fire_delta > 0:
            reward += fire_delta * 10.0

        # Stronger containment bonus
        if current_mean_fire < 0.2:
            reward += 5.0

        self.previous_mean_fire = current_mean_fire
        self.state[0] = fire_layer
        self.current_step += 1

        done = (self.current_step >= self.max_steps)
        truncated = False

        return (
            self._get_obs(),
            reward,
            done,
            truncated,
            {}
        )

def run_multi_agent_experiment(
    num_agents,
    seed=None,
    timesteps=100000
):
    if seed is not None:
        np.random.seed(seed)
        torch.manual_seed(seed)

    env = MultiAgentSaudiEnv(
        num_agents=num_agents
    )

    policy_kwargs = dict(
        features_extractor_class=CustomCNN,
        features_extractor_kwargs=dict(
            features_dim=256
        )
    )

    model = PPO(
        "CnnPolicy",
        env,
        policy_kwargs=policy_kwargs,
        verbose=0,
        seed=seed
    )

    model.learn(
        total_timesteps=timesteps
    )

    fire_values = []

    obs, info = env.reset(seed=seed)

    for _ in range(200):
        action, _ = model.predict(obs)
        obs, reward, done, truncated, info = env.step(action)

        fire_values.append(
            np.mean(env.state[0])
        )

        if done:
            break

    return fire_values, env


In [ ]:
seeds = [1, 2, 3]
results_1_agent = []
envs_1_agent = []

print("Running 1-Agent Multi-Seed Experiments...")
for s in seeds:
    print(f"  Seed {s}...")
    fire_vals, env = run_multi_agent_experiment(
        num_agents=1, seed=s, timesteps=100000
    )
    results_1_agent.append(fire_vals)
    envs_1_agent.append(env)

results_1_agent = np.array(results_1_agent)
print("1-agent experiments completed.")


In [ ]:
results_3_agent = []
envs_3_agent = []

print("Running 3-Agent Multi-Seed Experiments...")
for s in seeds:
    print(f"  Seed {s}...")
    fire_vals, env = run_multi_agent_experiment(
        num_agents=3, seed=s, timesteps=100000
    )
    results_3_agent.append(fire_vals)
    envs_3_agent.append(env)

results_3_agent = np.array(results_3_agent)
print("3-agent experiments completed.")


In [ ]:
results_5_agent = []
envs_5_agent = []

print("Running 5-Agent Multi-Seed Experiments...")
for s in seeds:
    print(f"  Seed {s}...")
    fire_vals, env = run_multi_agent_experiment(
        num_agents=5, seed=s, timesteps=100000
    )
    results_5_agent.append(fire_vals)
    envs_5_agent.append(env)

results_5_agent = np.array(results_5_agent)
print("5-agent experiments completed.")


In [ ]:
# Compute mean and std curves across seeds
mean_1 = np.mean(results_1_agent, axis=0)
std_1 = np.std(results_1_agent, axis=0)

mean_3 = np.mean(results_3_agent, axis=0)
std_3 = np.std(results_3_agent, axis=0)

mean_5 = np.mean(results_5_agent, axis=0)
std_5 = np.std(results_5_agent, axis=0)

print("Statistical curves computed successfully.")


In [ ]:
plt.figure(figsize=(10,6))
steps = np.arange(len(mean_1))

# 1 Agent
plt.plot(steps, mean_1, label="1 Agent", color='blue')
plt.fill_between(steps, mean_1 - std_1, mean_1 + std_1, color='blue', alpha=0.2)

# 3 Agents
plt.plot(steps, mean_3, label="3 Agents", color='green')
plt.fill_between(steps, mean_3 - std_3, mean_3 + std_3, color='green', alpha=0.2)

# 5 Agents
plt.plot(steps, mean_5, label="5 Agents", color='red')
plt.fill_between(steps, mean_5 - std_5, mean_5 + std_5, color='red', alpha=0.2)

plt.xlabel("Step")
plt.ylabel("Mean Fire Intensity")
plt.title("Multi-Agent Wildfire Containment Scaling (Mean ± Std over 3 seeds)")
plt.legend()
plt.grid()
plt.show()


In [ ]:
final_metrics = {
    '1 Agent Mean': mean_1[-1],
    '1 Agent Std': std_1[-1],
    '3 Agent Mean': mean_3[-1],
    '3 Agent Std': std_3[-1],
    '5 Agent Mean': mean_5[-1],
    '5 Agent Std': std_5[-1]
}

final_metrics


In [ ]:
print("Final Statistical Metrics (End of Episode):")
for key, value in final_metrics.items():
    print(f"{key}: {value:.4f}")


In [ ]:
# Visualize final state from the first seed of 5-agent configuration
env_vis = envs_5_agent[0]
fire_layer = env_vis.state[0]

plt.figure(figsize=(7,7))
plt.imshow(fire_layer)

for idx, (x, y) in enumerate(env_vis.agent_positions):
    plt.scatter(
        y,
        x,
        s=200,
        marker='X',
        label=f'Agent {idx+1}'
    )

plt.legend()
plt.title("5-Agent Wildfire Suppression (Seed 1)")
plt.colorbar()
plt.show()
